# 🫀 CardioTwin — 1D-CNN Arrhythmia Classification on Raw PPG
### Digital Twin Challenge 2026 (Happiest Health)

This notebook trains a **1-Dimensional Convolutional Neural Network (1D-CNN)** directly on **raw 100 Hz PPG waveform segments** (1000 samples per window, normalized).

**Hardware Acceleration:** Make sure to select **Runtime → Change runtime type → T4 GPU** in Google Colab.

**Safety Net Note:** Our XGBoost model is already trained and shipping. This deep learning model is an upgrade candidate targeting **85-92% accuracy**.

In [ ]:
# 1. Environment & Dependencies
!pip install -q wfdb scikit-learn matplotlib seaborn
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print('TensorFlow Version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

## 2. Load or Download Preprocessed 100 Hz PPG Segments
You can upload `ppg_100hz_dataset.npz` directly to the Colab files panel on the left, or download from your repository.

In [ ]:
# Load preprocessed segments
try:
    data = np.load('ppg_100hz_dataset.npz')
    X = data['X'] # Shape: (N, 1000, 1)
    y = data['y'] # Integer labels (0 to num_classes-1)
    classes = data['classes']
    print(f'Loaded dataset: {X.shape[0]} windows, {X.shape[1]} samples each.')
    print('Classes:', classes)
except Exception as e:
    print('Please upload ppg_100hz_dataset.npz to the Colab files tab on the left.')
    print('Error:', e)

## 3. 1D-CNN Deep Learning Architecture
Designed specifically for photoplethysmography (PPG):
- Wide initial receptive field (kernel 15) to capture morphological dicrotic notch and systolic peak.
- Batch normalization after each conv block for fast gradient flow.
- Global Average Pooling to prevent spatial overfitting and maintain rotational invariance.

In [ ]:
def build_1d_cnn(input_shape=(1000, 1), num_classes=6):
    inputs = layers.Input(shape=input_shape)
    
    # Block 1
    x = layers.Conv1D(32, kernel_size=15, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    
    # Block 2
    x = layers.Conv1D(64, kernel_size=11, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    
    # Block 3
    x = layers.Conv1D(128, kernel_size=7, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.3)(x)
    
    # Global Average Pooling
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name='CardioTwin_1DCNN')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_1d_cnn(input_shape=(1000, 1), num_classes=len(classes))
model.summary()

## 4. Train with Early Stopping & Learning Rate Reduction

In [ ]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=35,
    batch_size=32,
    callbacks=cb_list
)

## 5. Model Evaluation & Confusion Matrix

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print('=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, target_names=classes))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=classes)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('CardioTwin 1D-CNN Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix_1dcnn.png', dpi=200)
plt.show()

## 6. Export Trained Model for CardioTwin Backend Integration

In [ ]:
model.save('cardiotwin_1dcnn.keras')
print('Saved trained 1D-CNN to cardiotwin_1dcnn.keras!')
# Download to your local machine
from google.colab import files
files.download('cardiotwin_1dcnn.keras')